# 3. Tijd — trek de cyclus af, behoud het residu

**De centrale vraag die dit notebook beantwoordt: wat blijft er over zodra je corrigeert voor de kalender?**

Onder vrijwel elk tijdstempel van een bericht schuilt een dagelijkse en wekelijkse cyclus: mensen zijn overdag wakker en slapen 's nachts, zitten op weekdagen achter hun toetsenbord en zijn in het weekend elders. Het vinden van die cyclus is zélf niet het inzicht — *"er is een dagritme"* is geen nieuws en niemand heeft een dataset nodig om dat te ontdekken. Het is het **voorspelbare, saaie deel**, en de enige functie ervan is om te worden afgetrokken, zodat wat er overblijft (*het residu*) leesbaar en betekenisvol wordt.

We behandelen hier twee showcases die exact dezelfde beweging maken:

1. **Vliegtuigpassagiers (Flights)** — ruwe data vs. voortschrijdend gemiddelde (*rolling*) vs. overmatig afgevlakt, en de leugen van een rechte lijn die een gat in de index je vertelt. Klein, overzichtelijk en met de hand te verifiëren.
2. **De Ubuntu IRC-kanalen** — modelleer de dagelijkse cyclus, trek deze af en onderzoek hoe een release-dag eruitziet zodra je corrigeert voor het *gewone* patroon van de dag. Dát is de winst: een effect dat onzichtbaar is in de ruwe tijdreeks én onzichtbaar in de cyclus zelf, maar pas opduikt zodra je het basisniveau normaliseert en vormen vergelijkt.


In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    FlagDates,
    GroupAgg,
    MapValues,
    Pipeline,
    Share,
    SortValues,
    SubtractBaseline,
    TimeFeatures,
    TransformBase,
)
from goad_toolkit.visualizer import (
    BarPlot,
    FacetPlot,
    GroupedBarPlot,
    HistogramPlot,
    HorizontalLine,
    LinePlot,
    PlotSettings,
)

from wa_analyzer.data import load_own_chat, load_showcase

## 3.1 De showcase: vliegtuigpassagiers (flights)

Maandelijkse aantallen vliegtuigpassagiers, 1949–1960. Een duidelijke opwaartse trend, een jaarlijkse seizoenscyclus die meegroeit *met* de trend (multiplicatief, niet additief — de uitslagen worden groter naarmate het niveau stijgt), en geen ontbrekende maanden. Compact genoeg om met het blote oog te beredeneren.


In [ ]:
flights = load_showcase("flights")
flights["date"] = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"], format="%Y-%b")
flights = flights.set_index("date").sort_index()[["passengers"]]
flights.head()


In [ ]:
settings = PlotSettings(figsize=(10, 4), title="Monthly passengers, 1949-1960")
fig, ax = LinePlot(settings).plot(data=flights.reset_index(), x="date", y="passengers")

### Ruw, afgevlakt en overmatig afgevlakt — op één as

Een voortschrijdend venster (*rolling window*) dat breed genoeg is om er visueel 'strak' uit te zien, is vaak ook breed genoeg om precies dátgene weg te poetsen waarnaar je zocht. Er bestaat geen vensterbreedte die simpelweg "de juiste" is — een venster van 3 maanden raakt de ruis nauwelijks aan, terwijl een venster van 12 maanden (één volledige jaarcyclus) de *seizoensinvloed zelf* verwijdert, in plaats van alleen de toevallige fluctuaties. Plot ruwe data en afgevlakte curves daarom altijd samen op één as, zodat de lezer direct ziet wat het venster met de werkelijkheid heeft gedaan.

Twee `.rolling()`-aanroepen op één dataframe is al het datawerk dat hier nodig is. Een pipeline-stap verdient zijn plek pas wanneer het een beslissing is die het waard is om een naam te geven en te hergebruiken; "het gemiddelde over drie maanden" is gewoon een beknopte Pandas-regel totdat het een herbruikbare transformatie wordt.


In [ ]:
windows = pd.DataFrame({
    "raw": flights["passengers"],
    "3-month": flights["passengers"].rolling(3).mean(),
    "12-month": flights["passengers"].rolling(12).mean(),
})

smoothing = PlotSettings(figsize=(10, 4), title="One series, two windows", ylabel="passengers")
fig, ax = LinePlot(smoothing).plot(
    data=windows, dashes=False,
    palette={"raw": "#cccccc", "3-month": "steelblue", "12-month": smoothing.highlight_color},
)

Het 12-maands venster filtert niet zomaar ruis uit de jaarcyclus — het **is** exact één volledige cyclus. Door hierover te middelen wis je het seizoenspatroon zelf uit, in plaats van de ruis eromheen. Als jouw inhoudelijke claim ging over het jaarlijkse patroon, heeft dit venster geruisloos je eigen bewijsmateriaal gewist.


### Een gat in de index is niet hetzelfde als een gat in de grafiek

Sla een gat in het midden van de tijdreeks — laat vier maanden weg, alsof die periode van data nooit is geregistreerd — en plot op twee manieren: naïef (rechtstreeks), en na het herindexeren (*reindexing*) naar het volledige maandbereik.


In [ ]:
holey = flights.drop(flights.index[60:64])  # 1954, four months deliberately missing

full_index = pd.date_range(flights.index.min(), flights.index.max(), freq="MS")
reindexed = holey.reindex(full_index)

gaps = PlotSettings(
    figsize=(13, 4),
    title="Four missing months, plotted two ways",
    subplot_titles=["Plotted naively: the gap is invisible",
                    "Reindexed to the full range: the gap is a gap"],
)
host = LinePlot(gaps)
fig, axes = host.create_figure(n_plots=2)

# ax.plot, not LinePlot: seaborn's lineplot drops every row with a missing value before it
# draws, and has no option to keep them, so it would connect straight across the hole the
# right panel exists to show. matplotlib breaks the line at a NaN; so does pandas' .plot().
axes[0].plot(holey.index, holey["passengers"], marker="o", markersize=3)
axes[1].plot(reindexed.index, reindexed["passengers"], marker="o", markersize=3)

Het linkerpaneel tekent vol zelfvertrouwen een diagonale lijn dwars door vier maanden die helemaal niet bestaan — continuïteit (*continuity*) is een krachtig Gestalt-principe, en hier bedriegt het je op een gevaarlijke manier. Door `pd.date_range` te combineren met `.reindex()` bestaan de ontbrekende rijen weer als `NaN` (ontbrekende waarden), en Matplotlib onderbreekt de lijn netjes bij een `NaN` in plaats van er doorheen te verbinden. Een afwezigheid ziet er dan daadwerkelijk uit als een afwezigheid.

Dit is tevens de enige figuur in deze les die getekend is met `ax.plot` in plaats van `LinePlot`, en de reden daarvoor is de les zelf: Seaborn's `lineplot` verwijdert elke rij met een ontbrekende waarde vóórdat het tekent — een vaste eigenschap van die library, geen instelling die je kunt uitzetten. Daardoor zou Seaborn het gat dat het rechterpaneel juist moet laten zien doodleuk weer overbruggen! Matplotlib breekt een lijn netjes af bij een `NaN`, en Pandas' eigen `.plot()` doet dat ook. Weet wat jouw visualisatielaag met ontbrekende data doet vóórdat je erop vertrouwt om gaten zichtbaar te maken.

Of je die `NaN`'s vervolgens moet opvullen (met `0`, een interpolatie of een rolling schatting) is een aparte methodologische beslissing die je pas neemt nadat je het eerlijke gat hebt gezien — niet ervoor.


## 3.2 Inladen van de IRC showcase

Hetzelfde corpus als in les 1 en 2, dezelfde parsing — ditmaal netjes geïmporteerd en niet opnieuw vanaf nul opgebouwd.
`build_irc_pipeline()` levert al `day_name` op (via de `TimeFeatures`-stap uit les 1); de regex-features die het toevoegt (`has_url`, `n_question`, `addressed_to`) gebruiken we hier niet, net als in les 2. Dit notebook heeft nog twee kolommen nodig die de basis-pipeline niet maakt: een volledige `timestamp` (de parser houdt `date` en de gehele getallen voor `hh`/`mm` gescheiden) en een kolom `hour`.

Beide zijn pipeline-stappen, dus ze worden **toegevoegd aan de geïmporteerde pipeline** in plaats van achteraf handmatig op het dataframe geplakt: `BuildTimestamp` — gedefinieerd in `scripts/pipelines.py` naast de parser — plakt `date + hh:mm` aan elkaar, en `TimeFeatures` (dezelfde transformatie uit les 1) leidt daar `hour` uit af. Het uitbreiden van andermans pipeline met twee `.add(...)`-aanroepen is precies de kracht van pipelines als objecten, en exact wat je in 01.3 deed met je eigen chatdata.


In [ ]:
from scripts.pipelines import BuildTimestamp, build_irc_pipeline

pipeline = build_irc_pipeline()
pipeline.add(BuildTimestamp)
pipeline.add(TimeFeatures, name="clock", column="timestamp", features=["hour"])

msgs = pipeline.apply(load_showcase("ubuntu_irc"))
print(f"{len(msgs):,} messages across {msgs['channel'].nunique()} channels")
msgs.head()

### De voorspelbare cyclus

`FacetPlot` is de tool voor *"dezelfde grafiek, eenmaal per categorie"* — de *small-multiples* loop uit 2.1.2, gebundeld in één enkele aanroep: één paneel per weekdag, elk paneel getekend door dezelfde `HistogramPlot`. Daardoor zijn zowel het dagritme als de verschillen tussen maandag en zondag in één oogopslag zichtbaar, zonder zeven losse cellen te hoeven schrijven. `sharey=True` in de instellingen zorgt ervoor dat de panelen onderling vergelijkbaar zijn: als je Matplotlib per paneel zelf de schaal laat bepalen, zou een rustige zondag er net zo druk uitzien als een hectische dinsdag.


In [ ]:
uk = msgs[msgs["channel"] == "#ubuntu-uk"]
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

weekdays = PlotSettings(
    figsize=(12, 6),
    title="#ubuntu-uk: message volume by hour, one panel per weekday",
    xlabel="hour",
    max_cols=4,
    sharey=True,
)
fig, axes = FacetPlot(weekdays).plot(
    inner=HistogramPlot(weekdays), data=uk, by="day_name", order=order,
    x="hour", bins=range(25), stat="count", color="steelblue",
)

Dat is de cyclus, en die bevestigt de twee vanzelfsprekende zaken die een cyclus altijd laat zien: mensen slapen 's nachts en weekenden zien er anders uit (meer daarover in 3.4). Geen van beide is het werkelijke inzicht — het is slechts het referentiekader waar een specifieke dag tegen afgezet moet worden (*subtract the cycle*) voordat het ongebruikelijke zichtbaar kan worden.


## 3.3 Een wereldwijd evenement doorbreekt de lokale tijd

*Verandert een Ubuntu-release de vorm van de dag, of alleen het totale volume?*

`#ubuntu-uk` en `#ubuntu-it` zijn regionale kanalen: verschillende landen, verschillende normale werktijden en een verschillend piekuur. Een Ubuntu-release is echter een wereldwijde gebeurtenis — hetzelfde moment, overal tegelijk. Maakt het beide kanalen simpelweg drukker (meer berichten, maar dezelfde vorm), of verandert het *wanneer* mensen online zijn?

Tien releases vallen binnen het datumbereik van dit corpus (2013–2017), en — heel belangrijk om vooraf op te merken — **elk van die releases vond plaats op een donderdag**. Ubuntu releaste in deze periode consequent op donderdagen. Dat is cruciaal: als je een release-dag zou vergelijken met "alle andere dagen", vergelijk je stilzwijgend een donderdag met een baseline die ook weekenden bevat. In les 3.4 zullen we zien dat `#ubuntu-uk` zelf al een enorm verschil kent tussen weekdagen en weekenden. De enige eerlijke vergelijking is: release-donderdagen afzetten tegen *gewone* donderdagen.


In [ ]:
release_dates = pd.to_datetime([
    "2013-04-25", "2013-10-17",  # 13.04, 13.10
    "2014-04-17", "2014-10-23",  # 14.04, 14.10
    "2015-04-23", "2015-10-22",  # 15.04, 15.10
    "2016-04-21", "2016-10-13",  # 16.04, 16.10
    "2017-04-13", "2017-10-19",  # 17.04, 17.10
])
release_weekdays = release_dates.day_name().unique().tolist()  # ty: ignore[unresolved-attribute]
print(f"release weekdays in this window: {release_weekdays}")


### Gewone ritmes samengenomen — een grote, stabiele steekproef

Met ~250 gewone donderdagen per kanaal is het veilig om alle berichten samen te voegen tot één uurtotaal: geen enkele individuele dag kan een steekproef van 250 dagen domineren.

Elke databeslissing op weg naar de grafiek is een pipeline-stap met een duidelijke naam. `FlagDates` markeert de tien release-dagen — het werkpaard voor event-studies: markeer de data eenmalig met een vlaggetje en filter of groepeer vervolgens op die vlag. De aandelen voor `#ubuntu-uk` bestaan uit drie stappen: `Filter` naar gewone donderdagen (`day_name` zit al in de dataset via `TimeFeatures`), `CountValues(normalize=True)` voor de uuraandelen, en `SortValues` zodat de uren netjes in klokvolgorde worden geplot. Het `#ubuntu-it`-corpus komt binnen als uurtotalen in plaats van losse berichten, waardoor die pipeline in het midden verschilt: `GroupAgg` om de aantallen per uur op te tellen, en `Share` om ze te normaliseren — exact dezelfde beslissingen, vastgelegd als herbruikbare stappen.


In [ ]:
uk = (
    Pipeline()
    .add(FlagDates, column="date", dates=release_dates, feature="is_release")
    .apply(uk)
)

uk_share = (
    Pipeline()
    .add(Filter, expr="day_name == 'Thursday' and not is_release")
    .add(CountValues, column="hour", feature="share", normalize=True)
    .add(SortValues, column="hour")
    .apply(uk)
)

release_hourly = load_showcase("ubuntu_irc_release_hourly")
it_share = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-it' and not is_release")
    .add(GroupAgg, by="hour", column="messages", agg="sum")
    .add(Share, column="messages", feature="share")
    .apply(release_hourly)
)

rhythms = PlotSettings(
    figsize=(13, 4),
    title="Ordinary Thursdays, two channels",
    subplot_titles=[
        f"#ubuntu-uk, peak hour: {uk_share.loc[uk_share.share.idxmax(), 'hour']}",
        f"#ubuntu-it, peak hour: {it_share.loc[it_share.share.idxmax(), 'hour']}",
    ],
    xlabel="hour (UTC)",
    ylabel="share of messages",
)

host = BarPlot(rhythms)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(BarPlot(rhythms), axes[0], data=uk_share, x="hour", y="share",
                  color="steelblue")
host.plot_on_axes(BarPlot(rhythms), axes[1], data=it_share, x="hour", y="share",
                  color="darkorange")

Twee kanalen, twee verschillende normale ritmes — `#ubuntu-uk` piekt halverwege de ochtend (Britse tijd), `#ubuntu-it` piekt in de namiddag/avond. Volledig verwacht: verschillende landen, verschillende dagindelingen.

### Release-dagen — een steekproef van 9 tot 10 dagen, waar optellen misleidt

Er zijn slechts negen of tien release-donderdagen per kanaal. Tel je de ruwe aantallen simpelweg bij elkaar op, dan kan één uitzonderlijk drukke release het totaal volledig domineren — exact hetzelfde "wegpoetsen van het inzicht" als bij het rolling window in 3.1, maar dan een niveau hoger. Controleer dit altijd eerst, in plaats van blind te vertrouwen op een gemiddelde. De herstructurering die hiervoor nodig is — één rij per dag, één kolom per uur — wordt gedaan door `DayHourTable` (uit `scripts/profiles.py`), en dezelfde tabel voedt alle onderstaande analyses:


In [ ]:
from scripts.profiles import DayHourTable

release_days = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-it' and is_release")
    .add(DayHourTable, values="messages")
    .apply(release_hourly)
)

pd.DataFrame({
    "peak_hour": release_days.idxmax(axis=1),
    "total_messages": release_days.sum(axis=1),
}).sort_values("total_messages", ascending=False)

De drukste release-dag veruit (25 april 2013, Ubuntu 13.04) piekt op een heel ander uur dan de meeste andere releases, en met bijna het drievoudige volume van een typische release-dag. Bij een simpele optelling van ruwe berichten zou deze ene dag eigenhandig bepalen waar de "release-piek" valt. 

De oplossing: normaliseer elke dag eerst naar zijn eigen *uuraandeel* (relatief percentage), zodat een rustige release-dag en een extreem drukke release-dag even zwaar meewegen, en middelen pas daarna die aandelen. Elke release telt één keer mee, niet één keer per bericht.

Die oplossing vormt het **model**: het normale profiel van een gewone donderdag, op exact dezelfde wijze gemiddeld. Het **residu** is de release-dag minus dat model, uur voor uur — niet twee overlappende curven die een lezer met het blote oog moet aftrekken, maar de aftreksommen reeds uitgevoerd en geplot rond een horizontale nullijn. 
* Positief betekent: *"meer activiteit dan op een gewone donderdag op dit specifieke uur"*.
* Negatief betekent: *"minder activiteit dan normaal"*. 
De reguliere cyclus is er al volledig uitgefilterd.

In code is deze hele beweging één pipeline per kanaal: `Filter` naar de dagen waar de claim over gaat, `DayHourTable` om te transformeren, daarna het model — `MeanOfDayShares`, een transformatie die *in dit notebook zelf* is gedefinieerd omdat het de inhoudelijke aanname is die je moet kunnen verdedigen — en tenslotte `SubtractBaseline`, die het normale donderdagprofiel koppelt en het netto verschil overlaat. Filteren, transformeren, modelleren, aftrekken: de analyse leest exact als de vier methodologische beslissingen waaruit hij is opgebouwd.


In [ ]:
class MeanOfDayShares(TransformBase):
    """Average each day's own hourly share, so no single day's volume dominates."""

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        shares = data.div(data.sum(axis=1), axis=0)
        return shares.mean(axis=0).rename("share").rename_axis("hour").reset_index()


def profile_pipeline(expr: str, values: str | None = None) -> Pipeline:
    """Filter, reshape, model: the same three steps for all four profiles."""
    return (
        Pipeline()
        .add(Filter, expr=expr)
        .add(DayHourTable, values=values)
        .add(MeanOfDayShares)
    )


# The model: each channel's own ordinary-Thursday shape.
uk_ord_profile = profile_pipeline("day_name == 'Thursday' and not is_release").apply(uk)
it_ord_profile = profile_pipeline(
    "channel == '#ubuntu-it' and not is_release", values="messages"
).apply(release_hourly)

# The residual: the release-day pipeline ends by subtracting that model, so what
# flows out is the deviation, not the cycle plus the deviation.
uk_diff = (
    profile_pipeline("is_release")
    .add(SubtractBaseline, column="share", baseline=uk_ord_profile, on="hour",
         feature="difference")
    .apply(uk)
)
it_diff = (
    profile_pipeline("channel == '#ubuntu-it' and is_release", values="messages")
    .add(SubtractBaseline, column="share", baseline=it_ord_profile, on="hour",
         feature="difference")
    .apply(release_hourly)
)

residuals = pd.concat([
    uk_diff.assign(channel="#ubuntu-uk"),
    it_diff.assign(channel="#ubuntu-it"),
])

release_residual = PlotSettings(
    figsize=(9, 4),
    title="Release day minus an ordinary Thursday, by hour",
    xlabel="hour (UTC)",
    ylabel="share of that day's messages, release − ordinary",
)
lines = LinePlot(release_residual)
fig, ax = lines.plot(
    data=residuals, x="hour", y="difference", hue="channel",
    palette={"#ubuntu-uk": "steelblue", "#ubuntu-it": "darkorange"}, marker="o",
)
lines.plot_on(HorizontalLine(release_residual), y=0)

uk_peak = uk_diff.loc[uk_diff.difference.idxmax()]
it_peak = it_diff.loc[it_diff.difference.idxmax()]
print(f"#ubuntu-uk residual peak: {int(uk_peak.hour)}:00 UTC (+{uk_peak.difference:.1%})")
print(f"#ubuntu-it residual peak: {int(it_peak.hour)}:00 UTC (+{it_peak.difference:.1%})")

Beide residuen slaan op exact hetzelfde uur om naar positief — **14:00 UTC** — op een dag die er verder uitziet als willekeurige ruis rond de nullijn. `#ubuntu-it` schiet daar direct fors omhoog (+7,9 procentpunt) voordat het weer inzakt; `#ubuntu-uk` stijgt geleidelijker en blijft gedurende de hele middag (14:00–19:00) hoog voordat het wegebt. `#ubuntu-it` vertoont bovendien een tweede, afzonderlijke opleving in de avond (19:00–21:00) die `#ubuntu-uk` niet deelt — de twee kanalen doen dus niet exact hetzelfde, maar ze beginnen wél op exact hetzelfde klokuur. 

Die gezamenlijke start is het echte inzicht: een wereldwijd evenement maakt niet iedereen op zijn eigen lokale piekuur wat drukker (de afgetrokken cyclus toont dat direct aan); het voegt activiteit toe op één gedeeld globaal moment, en wat er daarna gebeurt hangt af van de lokale gewoonten van het kanaal.

### De hypothese om te toetsen: blijft men 's nachts op?

De voor de hand liggende aanname — dat mensen tot diep in de nacht opblijven voor een release — voorspelt *meer* nachtelijke activiteit op release-dagen. Toets dat op dezelfde manier: bereken voor elke dag het aandeel tussen 01:00 en 06:00, en vergelijk gewone donderdagen met release-dagen. Een simpele staafgrafiek met twee gemiddelden is hier echter niet genoeg. Gewone donderdagen variëren immers van nature — de meeste nachten gebeurt er niets, maar soms loopt een gesprek flink uit — en tien release-dagen is een kleine steekproef. 

De vraag is niet *"verschoof het gemiddelde?"*, maar *"verschoof het verder dan wat gewone donderdagen onderling al schommelen?"*. Daarom krijgt de grafiek de rijen per dag in plaats van alleen gemiddelden, en tekent Seaborn een **95%-betrouwbaarheidsinterval** op elke staaf: het bereik waar het gemiddelde in zou vallen als je opnieuw zo'n reeks dagen zou trekken. Een staaf zonder betrouwbaarheidsinterval is een bewering zonder spreiding, waardoor de lezer een echt inzicht niet kan onderscheiden van puur toeval.


In [ ]:
def night_shares(source: pd.DataFrame, expr: str, values: str | None = None) -> pd.Series:
    """One number per day: that day's own share of messages sent 01:00-06:00 UTC."""
    table = Pipeline().add(Filter, expr=expr).add(DayHourTable, values=values).apply(source)
    shares = table.div(table.sum(axis=1), axis=0)
    return shares.reindex(columns=range(1, 7), fill_value=0).sum(axis=1)


night = pd.concat([
    night_shares(uk, "day_name == 'Thursday' and not is_release")
    .rename("share").reset_index().assign(channel="#ubuntu-uk", day="ordinary Thursday"),
    night_shares(uk, "is_release")
    .rename("share").reset_index().assign(channel="#ubuntu-uk", day="release day"),
    night_shares(release_hourly, "channel == '#ubuntu-it' and not is_release", values="messages")
    .rename("share").reset_index().assign(channel="#ubuntu-it", day="ordinary Thursday"),
    night_shares(release_hourly, "channel == '#ubuntu-it' and is_release", values="messages")
    .rename("share").reset_index().assign(channel="#ubuntu-it", day="release day"),
])
print(night.groupby(["channel", "day"]).share.agg(["count", "mean", "median"]).round(4))

nobody = PlotSettings(
    figsize=(8, 4),
    title="Does the night's share move on release day?",
    ylabel="share of the day's messages, 01:00-06:00 UTC",
    base_color="#cccccc",
    highlight_color="#c44e52",
)
fig, ax = GroupedBarPlot(nobody).plot(
    data=night, x="channel", y="share", hue="day",
    palette=[nobody.base_color, nobody.highlight_color],
    errorbar=("ci", 95), capsize=0.1,
)
ax.legend(title="", loc="upper left", bbox_to_anchor=(1.01, 1))

De gemiddelden verschuiven weliswaar — het nachtaandeel van `#ubuntu-it` verdubbelt bijna op een release-dag, terwijl dat van `#ubuntu-uk` meer dan halveert — maar de betrouwbaarheidsintervallen tonen aan dat geen van beide verschuivingen groter is dan de natuurlijke variatie tussen gewone donderdagen onderling. Beide intervallen van de release-dagen overlappen ruim met de intervallen van de gewone donderdagen ernaast.

Daar zijn twee duidelijke redenen voor, beide zichtbaar in de cijfers: tien dagen blijft een kleine steekproef, en nachtelijke activiteit is extreem scheef verdeeld — de medianen zijn slechts een fractie van de gemiddelden, omdat de meeste nachten nagenoeg stil zijn en een paar nachtelijke uitschieters het gemiddelde omhoog trekken. Eén drukke nacht kan het gemiddelde over tien dagen al doen verdubbelen.

Het eerlijke wetenschappelijke oordeel luidt dan ook niet *"niemand blijft op"*, maar: *"álsk er al mensen opblijven, zijn tien release-dagen niet genoeg om dat statistisch hard te maken"*. De middagverschuiving hierboven had geen betrouwbaarheidsinterval nodig om evident te zijn; het nachteffect, als het al bestaat, vereist een steekproefgrootte die deze dataset simpelweg niet heeft. Beide zijn valide onderzoeksresultaten, en het expliciet tonen van het interval is precies bedoeld om dat onderscheid te kunnen maken.

### Verifieer het mechanisme voordat je een oorzaak claimt

Het is verleidelijk om te roepen: *"14:00 UTC is het tijdstip waarop de officiële release-aankondiging per mail verstuurd wordt, dus dat is de oorzaak."* Verifieer dat eerst grondig in plaats van het zomaar aan te nemen. Kijk naar de werkelijke `Date:`-header op de `ubuntu-announce` mailinglijst voor de releases in deze periode:

| Release | Tijdstip aankondiging (UTC) |
| --- | --- |
| 13.04 | 12:07 |
| 14.04 | 17:09 |
| 16.04 | 16:17 |
| 17.04 | 13:09 |

Dat is een spreiding van twee tot vijf uur, en absoluut geen vast tijdstip om 14:00 uur — de mailinglijst-aankondiging verklaart de piek dus niet! De bevinding blijft overeind, maar in een eerlijkere, meer bescheiden vorm: **een wereldwijd event doorbreekt het lokale ritme op een release-dag**, op een moment dat niet simpelweg samenvalt met "wanneer de e-mail werd verzonden". Wat de werkelijke trigger was (een blogpost, sociale media, download-mirrors die live gingen of mond-tot-mondreclame) kan deze dataset niet bewijzen. De integere keuze is om dat eerlijk te benoemen, in plaats van een verklaring te verzinnen die toevallig mooi klinkt.


## 3.4 De weekenddip zit in één kanaal, en de vorm verklaart waarom

De *small multiples* in 3.2 hintten al op een verschil tussen weekdagen en weekenden. Is dat een algemeen verschijnsel, of specifiek voor één kanaal? Een verhoudingsgetal is pas een echt inzicht als je weet waar het uit is opgebouwd.


In [ ]:
nl = msgs[msgs["channel"] == "#ubuntu-nl"]

for name, channel_df in [("#ubuntu-uk", uk), ("#ubuntu-nl", nl)]:
    channel_df = channel_df.copy()
    channel_df["is_weekend"] = channel_df["date"].dt.dayofweek >= 5

    weekday_per_day = (~channel_df["is_weekend"]).sum() / channel_df.loc[~channel_df["is_weekend"], "date"].nunique()
    weekend_per_day = channel_df["is_weekend"].sum() / channel_df.loc[channel_df["is_weekend"], "date"].nunique()
    msg_ratio = weekend_per_day / weekday_per_day

    wd_authors = channel_df[~channel_df["is_weekend"]].groupby("date")["author"].nunique().mean()
    we_authors = channel_df[channel_df["is_weekend"]].groupby("date")["author"].nunique().mean()
    author_ratio = we_authors / wd_authors

    msg_per_author_ratio = (weekend_per_day / we_authors) / (weekday_per_day / wd_authors)

    print(f"{name}")
    print(f"  messages/day ratio (weekend:weekday) = {msg_ratio:.2f}")
    print(f"  distinct authors/day ratio           = {author_ratio:.2f}")
    print(f"  messages per author ratio             = {msg_per_author_ratio:.2f}")
    print()


De dip is niet universeel — `#ubuntu-nl` beweegt nauwelijks. In `#ubuntu-uk` halveert het volume in het weekend ruimschoots, en dat heeft **meer te maken met wie er online komt dan met hoeveel ieder individu zegt**: het aantal unieke auteurs daalt veel harder dan het aantal berichten per auteur.

Maar een ratio verklaart uit zichzelf nog niets; de vorm van de grafiek wél. Beide kanalen krijgen een eigen paneel, want de stelling *"de dip is specifiek voor `#ubuntu-uk`"* is een bewering over een *vergelijking*, en een vergelijking die de lezer niet met eigen ogen kan zien moet blind geloofd worden. Eén enkele pipeline bouwt het profiel voor beide kanalen: `Filter` naar het kanaal, `MapValues` om de zeven daglabels samen te vouwen tot weekdag/weekend, `GroupAgg` om te tellen per (periode, uur), en `Share(by="period")` zodat de uren van elke periode apart optellen tot 100%. Daardoor worden een rustig weekend en een drukke werkweek puur op basis van hun *vorm* vergelijkbaar — en dat is waar het om draait:


In [ ]:
def period_profile(channel: str) -> pd.DataFrame:
    """Hour shares per period, each period summing to one on its own."""
    return (
        Pipeline()
        .add(Filter, expr=f"channel == '{channel}'")
        .add(MapValues, column="day_name",
             mapping={"Saturday": "weekend", "Sunday": "weekend"},
             feature="period", default="weekday")
        .add(GroupAgg, by=["period", "hour"], agg="size")
        .add(Share, column="n", by="period", feature="share")
        .apply(msgs)
    )


weekend = PlotSettings(
    figsize=(13, 4),
    title="Only #ubuntu-uk trades the office for the evening",
    subplot_titles=["#ubuntu-uk", "#ubuntu-nl"],
    xlabel="hour (UTC)",
    ylabel="share of messages",
    sharey=True,
)
host = LinePlot(weekend)
fig, axes = host.create_figure(n_plots=2)
for ax, channel in zip(axes, ["#ubuntu-uk", "#ubuntu-nl"]):
    host.plot_on_axes(
        LinePlot(weekend), ax, data=period_profile(channel), x="hour", y="share",
        hue="period", palette={"weekday": "steelblue", "weekend": "crimson"},
        marker="o", markersize=3,
    )

Op weekdagen piekt `#ubuntu-uk` halverwege de ochtend (Britse tijd) en neemt de activiteit gedurende de dag geleidelijk af. In het weekend draait dit patroon volledig om: de hele ochtend is het stil, waarna de piek pas in de avond valt. **Het verhaal hierachter: mensen chatten onder werktijd.** De weekdagpiek valt samen met de start van de Britse kantooruren; in het weekend verschijnt dezelfde community pas 's avonds vanaf de thuisbank.

`#ubuntu-nl` fungeert hier als het controlekanaal dat dit verhaal geloofwaardig maakt. Het profiel van `#ubuntu-nl` piekt op weekdagen al standaard in de avond — het was van meet af aan nooit een 'kantoorkanaal' — en de weekendlijn wijkt daar nauwelijks vanaf. Verkeer dat nooit afhankelijk was van de werkdag heeft immers niets te verliezen wanneer het weekend aanbreekt. Waarom twee nationale kanalen zulke verschillende gewoonten ontwikkelden, kan deze dataset niet met zekerheid zeggen — een andere community, andere culturele normen rondom chatten op het werk, of toevallig de gewoonten van de vaste kern. Maar dat hoeft ook niet: het *contrast* draagt het inzicht. Het kanaal met de kantoortijden-piek is het kanaal met de kantoortijden-dip.

Wat dit verhaal wetenschappelijk overtuigend maakt in plaats van een toevalligheid verpakt als leuk verhaaltje:
1. De piek valt op logische werktijden voor de betreffende tijdzone, niet op een willekeurig uur.
2. De activiteit verdwijnt in het weekend niet zomaar, maar *verschuift* van de ochtend naar de avond.
3. De richting van het effect was voorspelbaar vóórdat we de data bekeken.
4. Het kanaal zonder kantoortijden-patroon vertoont ook géén weekenddip.


## 3.5 Suggesties voor je eigen chat

Dezelfde techniek die hierboven de release-dag en de weekendverschuivingen blootlegde — tellen, groeperen, de cyclus aftrekken en naar de vorm kijken — kun je toepassen op jullie eigen WhatsApp-gesprek. Plotly's interactieve *area chart* is een waardevolle toevoeging aan je toolkit hiervoor: het is interactief, en cumulatieve tellingen maken een groeiverhaal in één oogopslag inzichtelijk.


In [ ]:
own = load_own_chat()

In [ ]:
import plotly.express as px

own = own.copy()
own["date"] = pd.to_datetime(own["timestamp"]).dt.date

daily = own.groupby(["date", "author"]).size().reset_index(name="message_count")  # ty: ignore[no-matching-overload]
daily["cumulative"] = daily.groupby("author")["message_count"].cumsum()

fig = px.area(
    daily, x="date", y="cumulative", color="author", line_group="author",
    labels={"cumulative": "cumulative messages"},
    title="Cumulative messages per author, over time",
)
fig.show()

### Wie begint er — twee auteurs, of een groep?

Met exact twee auteurs is de vraag *"wie stuurt het eerste bericht van de dag?"* een zuiver +1 / −1 signaal dat je cumulatief kunt optellen: een lopend saldo dat direct laat zien of één persoon structureel het gesprek opent. Met meer dan twee auteurs is die vraag niet langer binair — het alternatief voor groepsgesprekken hieronder hergebruikt het *small-multiples* principe uit 3.2: het uurgewijze ritme van elke auteur zij aan zij weergegeven.


In [ ]:
n_authors = own["author"].nunique()
print(f"{n_authors} authors in this chat")

In [ ]:
if n_authors == 2:
    daily_first = own.sort_values("timestamp").groupby("date")["author"].first()
    authors = sorted(daily_first.unique())
    balance = daily_first.map({authors[0]: 1, authors[1]: -1}).cumsum()  # ty: ignore[invalid-argument-type]

    first_mover = PlotSettings(
        figsize=(10, 4),
        title=f"Who sends the first message of the day?\n"
              f"+1 = {authors[0]}, -1 = {authors[1]}",
        ylabel="cumulative balance",
        xtick_rotation=45,
    )
    lines = LinePlot(first_mover)
    fig, ax = lines.plot(data=balance.rename("balance").reset_index(), x="date", y="balance")
    lines.plot_on(HorizontalLine(first_mover), y=0, linewidth=0.8)

**Belangrijke waarschuwing:** Deze grafieken zijn PAS interessant ALS je er een echte verhaallijn aan kunt koppelen: wat is er aan de hand, wat gebeurde er op specifieke momenten, kun je concrete gebeurtenissen (events) toevoegen die de data verklaren, en kun je achtergrondcontext bieden? Kun je dat niet, dan is dit mogelijk een volkomen nutteloze of nietszeggende grafiek en moet je op zoek gaan naar betekenisvollere patronen. Dit gedeelte is vooral bedoeld als inspiratie voor het type vragen dat je kunt stellen over de factor tijd.


In [ ]:
if n_authors > 2:
    pipeline = Pipeline()
    pipeline.add(TimeFeatures, column="timestamp", features=["hour"])
    own_enriched = pipeline.apply(own)

    rhythm = PlotSettings(
        figsize=(12, 6),
        title="Group chat: hourly rhythm per author",
        xlabel="hour",
        max_cols=4,
        sharey=True,
    )
    FacetPlot(rhythm).plot(
        inner=HistogramPlot(rhythm), data=own_enriched, by="author",
        x="hour", bins=range(25), stat="count", color="steelblue",
    )

## 3.6 Reflectievragen

1. **Wat is de voorspelbare cyclus in jullie eigen chat, en wat heb je ervan afgetrokken om het residu zichtbaar te maken?**
   Benoem eerst de cyclus (dagelijks, wekelijks) vóórdat je benoemt wat er overblijft — de volgorde is methodologisch essentieel.
2. **Heeft een rolling window ooit het inzicht weggepoetst waarnaar je op zoek was, in plaats van alleen de toevallige ruis eromheen?**
   Vermeld welk venster je hebt gekozen en motiveer waarom die specifieke breedte, en niet een bredere of smallere.
3. **Als je een plotselinge verschuiving vond (zoals de Ubuntu release-dag), heb je dan eerst gezocht naar een alledaagse verklaring vóór een wereldschokkende theorie?**
   Denk aan een gedeeld evenement, een feestdag, vakantie, of een verandering in wie er überhaupt aanwezig was.
4. **Welke inhoudelijke claim zou een niet-geherindexeerd gat in de tijdreeks voor je verborgen hebben gehouden?**
